# Forecast-Window Delta Analysis (Paper Figure 3)

This notebook reproduces the **paper's Figure 3 logic** using reusable package modules instead of inline function definitions.

## What This Notebook Does
1. Loads a standardized 4-degree model catalog and the paper-year policy.
2. Scans available yearly files to identify usable models and recommend one debug target.
3. Runs one model to produce a detailed window-delta table and a single-model heatmap.
4. Runs all models, exports matrix-style CSV outputs, and plots the 3-panel delta summary.

## Important Evaluation Rules
- Forecast windows: `1-5`, `6-10`, `11-15`, `16-20`, `21-25`, `26-30` days.
- Tolerances: `+/-2`, `+/-2`, `+/-3`, `+/-3`, `+/-5`, `+/-5` days.
- Delta definition: `Model - Climatology` (negative means improvement).
- CMZ mask: legacy polygon option enabled by default to match prior scripts.


In [1]:
from pathlib import Path

from IPython.display import display

from monsoonbench.metrics.cmz_window_metrics import WindowScoringConfig
from monsoonbench.utils.forecast_window_pipeline import (
    PAPER_YEARS_BY_MODEL_4DEG,
    build_model_run_config,
    choose_recommended_model,
    compute_window_delta_table,
    get_default_four_degree_model_specs,
    run_multi_model_window_analysis,
    save_window_analysis_outputs,
    scan_forecast_data_coverage,
)
from monsoonbench.visualization.cmz_window_plots import (
    plot_multi_model_window_deltas,
    plot_window_delta_heatmap,
)

## Step 1: Configure Shared Inputs

This block defines project paths and common runtime options.

- `base_config` controls climatology/model computation settings.
- `scoring_config` controls window classification behavior.
- `model_specs` is a reusable 4-degree catalog aligned with repository structure.


In [2]:
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()),
    Path.cwd(),
).parent

data_dir = repo_root / "data"
output_dir = repo_root / "monsoon-bench" / "examples" / "paper_figures" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

base_config = {
    "imd_folder": data_dir / "imd_rainfall_data" / "4p0", #Change as needed
    "thres_file": data_dir / "imd_onset_threshold" / "mwset4x4.nc4", #Change as needed
    "mok": True,
    "max_forecast_day": 30,
    "file_pattern": "{}.nc",
    "temporal_agg_mode": "year_then_average",
    "use_legacy_cmz_polygon": False,
    "strict_year_failures": False,
}

model_paths = {
    "IFS": f"{data_dir}/rainfall_4p0/IFS_S2S",
    "AIFS":  f"{data_dir}/rainfall_4p0/AIFS",
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
    "Graphcast": f"{data_dir}/rainfall_4p0/GraphCast",
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
    "FuXi-S2S": f"{data_dir}/rainfall_4p0/FuXi_S2S",
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}

scoring_config = WindowScoringConfig(
    obs_window_mode="bin",
    classification_mode="strict_window",
    fn_tn_obs_mode="bin",
    mr_denom_mode="obs_in_bin",
    aggregate_mode="gridmean",
)
model_specs = get_default_four_degree_model_specs(data_dir, model_paths)
print("Output directory:", output_dir)
print("Root directory:", repo_root)

Output directory: c:\Users\cflor\CSAssignments\Clinic3\monsoon-bench\examples\paper_figures\outputs
Root directory: c:\Users\cflor\CSAssignments\Clinic3


## Step 2: Data Coverage Check

Before running expensive computations, verify each model has the years required by the paper policy.
The recommended model is chosen by: paper-year support -> ensemble check -> number of available years.


In [3]:
coverage_df = scan_forecast_data_coverage(model_specs, PAPER_YEARS_BY_MODEL_4DEG)
display_cols = [
    "model",
    "model_type",
    "supports_paper_years",
    "ensemble_check_ok",
    "n_years",
    "min_year",
    "max_year",
    "missing_paper_years",
]

display(coverage_df[display_cols])

recommended = choose_recommended_model(coverage_df)
recommended_name = str(recommended["model"])
print("Recommended debug model:", recommended_name)

,model,model_type,supports_paper_years,ensemble_check_ok,n_years,min_year,max_year,missing_paper_years
1,aifs,deterministic,True,True,60,1965,2024,[]
2,fuxi,deterministic,True,True,60,1965,2024,[]
3,graphcast,deterministic,True,True,60,1965,2024,[]
6,ngcm,probabilistic,True,True,60,1965,2024,[]
5,fuxi-s2s,probabilistic,True,True,20,2002,2021,[]
4,gencast,probabilistic,True,True,20,1965,2024,[]
0,ifs,probabilistic,True,True,20,2004,2023,[]


Recommended debug model: aifs


## Step 3: Single-Model Walkthrough

Run one model end-to-end first. This is the best place to inspect:
- per-window MAE/FAR/MR deltas
- TP/FP/FN/TN counts
- processed vs skipped years

Change `target_model` if you want to debug a specific model directly.


In [ ]:
target_model = recommended_name
single_run = build_model_run_config(
    model_name="aifs",
    base_config=base_config,
    model_specs=model_specs,
    paper_years_by_model=PAPER_YEARS_BY_MODEL_4DEG,
    scoring=scoring_config,
)
single_delta = compute_window_delta_table(single_run)
display(
    single_delta[
        [
            "metric",
            "window",
            "model",
            "climatology",
            "delta",
            "model_tp",
            "model_fp",
            "model_fn",
            "model_tn",
        ]
    ]
)

single_plot_path = output_dir / f"forecast_window_skill_single_{target_model}.png"
_ = plot_window_delta_heatmap(
    single_delta, model_name=target_model, save_path=single_plot_path
)
print("Saved:", single_plot_path)

Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1904-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1905.nc
Renamed 

,metric,window,model,climatology,delta,model_tp,model_fp,model_fn,model_tn
0,MAE (days),1-6,2.485852,4.523810,-2.037957,72,49,23,675
1,FAR (%),1-6,6.337913,4.246591,2.091322,72,49,23,675
2,MR (%),1-6,25.833333,67.500000,-41.666667,72,49,23,675
3,MAE (days),5-10,3.690008,4.672884,-0.982876,43,50,58,668
4,FAR (%),5-10,6.521793,4.727701,1.794092,43,50,58,668
5,MR (%),5-10,54.166667,76.666667,-22.500000,43,50,58,668
6,MAE (days),10-15,5.399058,5.682937,-0.283879,40,66,65,648
7,FAR (%),10-15,8.648892,7.962480,0.686412,40,66,65,648
8,MR (%),10-15,60.833333,75.833333,-15.000000,40,66,65,648
9,MAE (days),16-21,7.002601,5.878704,1.123898,30,55,77,657


KeyError: "None of [Index(['1-5', '6-10', '11-15', '16-20', '21-25', '26-30'], dtype='object', name='window')] are in the [columns]"

: 

## Step 4: Batch Run for Paper Figure 3

This cell executes the full multi-model workflow and exports reusable artifacts:
- per-model long delta tables
- matrix-form CSV files for MAE/FAR/MR (+ model order list)
- combined long table for all models
- three-panel heatmap used for final comparison


In [ ]:
batch_result = run_multi_model_window_analysis(
    base_config=base_config,
    model_specs=model_specs,
    paper_years_by_model=PAPER_YEARS_BY_MODEL_4DEG,
    scoring=scoring_config,
)
saved_files = save_window_analysis_outputs(
    batch_result, output_dir=output_dir, prefix="forecast_window_skill"
)

multi_plot_path = output_dir / "forecast_window_skill_delta_panels.png"
_ = plot_multi_model_window_deltas(
    batch_result["delta_for_plot"],
    batch_result["plot_model_order"],
    batch_result["window_labels"],
    batch_result["plot_model_labels"],
    save_path=multi_plot_path,
)
print("Saved plot:", multi_plot_path)
print("Saved CSV files:")
for key, path in saved_files.items():
    print(f"- {key}: {path}")


Running model=ifs, years=[2019, 2020, 2021, 2022, 2023]
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1904-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssign

Exception ignored while calling weakref callback <function WeakValueDictionary.__init__.<locals>.remove at 0x0000020C21234BF0>:
Traceback (most recent call last):
  File "C:\Users\cflor\AppData\Local\Python\pythoncore-3.14-64\Lib\weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 


Processing init time 16/26: 2022-06-24
Processing init time 21/26: 2022-07-11
Processing init time 26/26: 2022-07-29

Processing Summary:
Total potential initializations: 312
Skipped (no observed onset): 0
Skipped (initialized after observed onset): 142
Valid initializations processed: 170
Ensemble onsets found (≥50% members): 121
Ensemble onset rate: 0.712
Note: Only onsets on or after 6/2 were counted due to MOK flag
Processing climatology as forecast for 26 init times x 3 lats x 4 lons...
Year: 2022
Only processing forecasts initialized before observed onset dates
Processing init time 1/26: 2022-05-02
Processing init time 6/26: 2022-05-20
Processing init time 11/26: 2022-06-06
Processing init time 16/26: 2022-06-24
Processing init time 21/26: 2022-07-11
Processing init time 26/26: 2022-07-29

Climatology Forecast Summary:
Total potential initializations: 312
Skipped (no observed onset): 0
Skipped (initialized after observed onset): 142
Valid initializations processed: 170
Onsets for